# 🛡️ LexGuard Bulk Ingestion v2 (Groq + HuggingFace)

This notebook uses:
- **Groq API** for HyDE scenario generation (cloud-hosted llama3, ~500 tokens/sec)
- **sentence-transformers** for local embeddings (runs on Colab CPU/GPU)
- **pymongo** to write directly to your MongoDB Atlas cluster

No Ollama needed. No local GPU required for LLM inference.

In [ ]:
!pip install -q pymongo PyPDF2 groq sentence-transformers

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Paste your keys here
# ══════════════════════════════════════════════════════════════

MONGODB_URI = "YOUR_MONGODB_ATLAS_CONNECTION_STRING"
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

# Folder where you uploaded your PDFs in Colab
PDF_FOLDER_PATH = "/content/bare_acts"

# Groq model for HyDE generation
GROQ_MODEL = "llama3-8b-8192"

# Embedding model (runs locally on Colab, no API needed)
EMBED_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

In [ ]:
import os
import re
import json
import time
import PyPDF2
from groq import Groq
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# ── Initialize Clients ──
groq_client = Groq(api_key=GROQ_API_KEY)
mongo_client = MongoClient(MONGODB_URI)
db = mongo_client.get_default_database()
collection = db['statutenodes']

# Load embedding model locally on Colab
print("Loading embedding model (one-time download)...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, trust_remote_code=True)
print(f"✅ Embedding model loaded. Dimension: {embed_model.get_sentence_embedding_dimension()}")
print(f"✅ Connected to MongoDB: {db.name}")
print(f"✅ Groq client ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CORE FUNCTIONS
# ══════════════════════════════════════════════════════════════

def get_embedding(text):
    """Generate embedding locally using sentence-transformers."""
    safe_text = text[:8000]  # nomic supports up to 8192 tokens
    embedding = embed_model.encode(safe_text, normalize_embeddings=True)
    return embedding.tolist()

def get_hyde_summary(section_text, retries=3):
    """Generate HyDE scenarios using Groq (cloud llama3)."""
    safe_text = section_text[:4000]
    prompt = (
        "You are a legal expert. I will provide you with a section of an Indian Bare Act. "
        "Generate 5 highly realistic, modern corporate scenarios (like non-competes, NDA breaches, "
        "software disputes, startup equity fights, SaaS contract violations) where this specific law "
        "would be the deciding factor. Output ONLY the 5 scenarios as a numbered list.\n\n"
        f"Section Text:\n{safe_text}"
    )

    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=1024
            )
            return response.choices[0].message.content
        except Exception as e:
            error_msg = str(e)
            if "rate_limit" in error_msg.lower() or "429" in error_msg:
                wait_time = 15 * (attempt + 1)  # 15s, 30s, 45s backoff
                print(f"      ⏳ Rate limited. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"      ⚠️ Groq error (attempt {attempt+1}): {error_msg[:100]}")
                time.sleep(5)
    return ""  # Return empty string if all retries fail

def process_pdf(filepath):
    """Parse a PDF and split by Sections/Chapters."""
    act_name = os.path.basename(filepath).replace('.pdf', '').replace('_', ' ')

    with open(filepath, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        text = "".join(page.extract_text() or "" for page in reader.pages)

    lines = text.split('\n')
    current_chapter = "Preliminary"
    current_section = None
    section_buffer = []
    chunks = []

    for line in lines:
        cleaned = line.strip()
        if not cleaned:
            continue

        # Check for Chapter
        chapter_match = re.match(r'^CHAPTER\s+([A-Z0-9]+)', cleaned, re.IGNORECASE)
        if chapter_match:
            current_chapter = f"Chapter {chapter_match.group(1)}"
            continue

        # Check for Section
        section_match = re.match(r'^(?:Section\s+)?(\d+[A-Z]?)\.\s+(.*)', cleaned, re.IGNORECASE)
        if not section_match:
            section_match = re.match(r'^(\d+[A-Z]?)\.\s+(.*)', cleaned)

        if section_match:
            if current_section and section_buffer:
                chunks.append({
                    "actName": act_name,
                    "chapter": current_chapter,
                    "sectionNumber": f"Section {current_section}",
                    "content": " ".join(section_buffer)
                })
            current_section = section_match.group(1)
            section_buffer = [section_match.group(2)]
        else:
            if current_section:
                section_buffer.append(cleaned)

    # Save last section
    if current_section and section_buffer:
        chunks.append({
            "actName": act_name,
            "chapter": current_chapter,
            "sectionNumber": f"Section {current_section}",
            "content": " ".join(section_buffer)
        })

    return chunks

In [ ]:
# ══════════════════════════════════════════════════════════════
# UPLOAD YOUR PDFs FIRST!
# On the left sidebar, click the folder icon.
# Create a folder called 'bare_acts' and upload all 47 PDFs.
# Then run this cell.
# ══════════════════════════════════════════════════════════════

if not os.path.exists(PDF_FOLDER_PATH):
    os.makedirs(PDF_FOLDER_PATH)
    print(f"❌ Created empty folder at {PDF_FOLDER_PATH}.")
    print(f"   Please upload your PDFs there and re-run this cell.")
else:
    pdf_files = sorted([f for f in os.listdir(PDF_FOLDER_PATH) if f.endswith('.pdf')])
    print(f"📂 Found {len(pdf_files)} PDFs to process.")
    for i, f in enumerate(pdf_files):
        print(f"   {i+1}. {f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# MAIN INGESTION LOOP
# This is the cell that does all the work.
# Groq free tier: ~30 requests/min for llama3-8b.
# We add a 2.5s delay between requests to stay safe.
# ══════════════════════════════════════════════════════════════

DELAY_BETWEEN_REQUESTS = 2.5  # seconds — adjust if you hit rate limits

pdf_files = sorted([f for f in os.listdir(PDF_FOLDER_PATH) if f.endswith('.pdf')])
total_files = len(pdf_files)
grand_total_sections = 0
grand_total_success = 0
start_time = time.time()

for file_idx, filename in enumerate(pdf_files):
    filepath = os.path.join(PDF_FOLDER_PATH, filename)
    sections = process_pdf(filepath)
    total_sections = len(sections)
    grand_total_sections += total_sections
    success_count = 0

    print(f"\n{'='*60}")
    print(f"📄 [{file_idx+1}/{total_files}] {filename}")
    print(f"   Extracted {total_sections} sections.")
    print(f"{'='*60}")

    for i, sec in enumerate(sections):
        try:
            # 1. HyDE Generation via Groq
            hyde_scenarios = get_hyde_summary(sec['content'])
            enriched_text = f"{sec['content']}\n\n[MODERN SCENARIOS]\n{hyde_scenarios}"

            # 2. Embedding via sentence-transformers (local, instant)
            embedding = get_embedding(enriched_text)

            # 3. Upsert to MongoDB Atlas
            collection.update_one(
                {"actName": sec['actName'], "sectionNumber": sec['sectionNumber']},
                {"$set": {
                    "actName": sec['actName'],
                    "sectionNumber": sec['sectionNumber'],
                    "content": enriched_text,
                    "domain": sec['chapter'],
                    "embedding": embedding
                }},
                upsert=True
            )
            success_count += 1
            grand_total_success += 1

            if (i + 1) % 5 == 0 or (i + 1) == total_sections:
                elapsed = time.time() - start_time
                rate = grand_total_success / (elapsed / 60) if elapsed > 0 else 0
                print(f"   ✅ {i+1}/{total_sections} sections | Total: {grand_total_success} | Rate: {rate:.1f} sections/min")

            # Rate limit delay
            time.sleep(DELAY_BETWEEN_REQUESTS)

        except Exception as e:
            print(f"   ⚠️ Failed {sec['sectionNumber']}: {str(e)[:120]}")
            time.sleep(5)  # Extra pause on error

    print(f"   ✅ Completed: {success_count}/{total_sections} sections saved.")

elapsed_total = time.time() - start_time
print(f"\n{'='*60}")
print(f"🎉 BULK INGESTION COMPLETE!")
print(f"   Total Sections Processed: {grand_total_success}/{grand_total_sections}")
print(f"   Total Time: {elapsed_total/60:.1f} minutes")
print(f"{'='*60}")